# Sequence Modeling

Build chronological next-step examples from a local synthetic series, verify their alignment, and compare a small model with persistence. The split is based on target time; no external data are needed.

In [1]:
import math
import torch
from torch import nn

torch.manual_seed(25)
torch.set_num_threads(1)
print('CPU | synthetic chronological one-step forecast')

CPU | synthetic chronological one-step forecast


## Generate a series and define target-time splits

The series combines a trend and periodic signal. Train, validation, and test targets occupy separate chronological regions. Context may include observations before each target, as it would at prediction time.

In [3]:
time = torch.arange(360, dtype=torch.float32)
series = 0.015 * time + torch.sin(time * (2 * math.pi / 30)) + 0.25 * torch.sin(time * (2 * math.pi / 9))
window = 12
train_end, valid_end = 220, 290

def make_windows(values, start_target, end_target, size):
    inputs, targets = [], []
    for target_index in range(max(size, start_target), end_target):
        inputs.append(values[target_index-size:target_index])
        targets.append(values[target_index])
    return torch.stack(inputs), torch.stack(targets)

train_x, train_y = make_windows(series, window, train_end, window)
valid_x, valid_y = make_windows(series, train_end, valid_end, window)
test_x, test_y = make_windows(series, valid_end, len(series), window)
assert train_x.shape[1] == window and train_y.shape[0] == train_x.shape[0]
torch.testing.assert_close(train_x[0], series[:window])
torch.testing.assert_close(train_y[0], series[window])
torch.testing.assert_close(train_y[-1], series[train_end - 1])
torch.testing.assert_close(valid_y[0], series[train_end])
torch.testing.assert_close(valid_y[-1], series[valid_end - 1])
torch.testing.assert_close(test_y[0], series[valid_end])
print('first input:', train_x[0].tolist())
print('first target:', train_y[0].item())
print('examples train/validation/test:', len(train_y), len(valid_y), len(test_y))

first input: [0.0, 0.3836085796356201, 0.6829385757446289, 0.8492916226387024, 0.8886499404907227, 0.8555204272270203, 0.8245501518249512, 0.8533199429512024, 0.9538249373435974, 1.0860563516616821, 1.1767222881317139, 1.1543467044830322]
first target: 0.9842914938926697
examples train/validation/test: 208 70 70


## Scale using training observations only

The mean and standard deviation come only from the training segment. Validation and test values are transformed using those fixed statistics.

In [4]:
mean = series[:train_end].mean()
scale = series[:train_end].std().clamp_min(1e-6)
train_x = (train_x - mean) / scale
train_y = (train_y - mean) / scale
valid_x = (valid_x - mean) / scale
valid_y = (valid_y - mean) / scale
test_x = (test_x - mean) / scale
test_y = (test_y - mean) / scale
assert torch.isfinite(train_x).all() and torch.isfinite(test_y).all()
print(f'train-only mean={mean.item():.4f}, scale={scale.item():.4f}')

train-only mean=1.6758, scale=1.1872


## Fit a one-step predictor and select by validation loss

A small feed-forward network consumes each fixed-length window. The persistence baseline predicts the final value in that window. Validation selects the checkpoint; the test set is held until after selection.

In [5]:
model = nn.Sequential(nn.Linear(window, 32), nn.Tanh(), nn.Linear(32, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
best_state, best_valid, best_epoch = None, float('inf'), None
for epoch in range(250):
    model.train()
    optimizer.zero_grad()
    train_loss = nn.functional.mse_loss(model(train_x).squeeze(-1), train_y)
    train_loss.backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        valid_loss = nn.functional.mse_loss(model(valid_x).squeeze(-1), valid_y).item()
    if valid_loss < best_valid:
        best_valid, best_epoch = valid_loss, epoch + 1
        best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
model.load_state_dict(best_state)
with torch.no_grad():
    prediction = model(test_x).squeeze(-1)
    persistence = test_x[:, -1]
    model_mse = nn.functional.mse_loss(prediction, test_y).item()
    baseline_mse = nn.functional.mse_loss(persistence, test_y).item()
print(f'validation-selected epoch={best_epoch}, validation MSE={best_valid:.4f}')
print(f'test MSE: model={model_mse:.4f}, persistence={baseline_mse:.4f}')

validation-selected epoch=240, validation MSE=0.0099
test MSE: model=0.1069, persistence=0.0248


The comparison uses a synthetic stationary pattern plus trend and a fixed chronological split. It demonstrates target alignment, train-only preprocessing, validation selection, and a baseline; it does not establish forecasting skill on real data. For multi-step horizons, evaluate each horizon separately and consider free-running error accumulation.